In [2]:
import pandas as pd
df = pd.read_csv('cleaned_dataset_halal_haram.csv')

# EDA & Data Cleaning

In [3]:
df.head(10)

,text,label
0,vegetable oil,halal
1,beef stock contains less than of mirepoix carr...,halal
2,clam stock potatoes clams cream vegetable oil ...,haram
3,water cream broccoli celery vegetable oil corn...,haram
4,chicken stock contains less than of yeast extr...,halal
5,water pea beans carrots cooked ham water added...,haram
6,diced tomatoes in tomato juice tomato puree wa...,halal
7,tomato puree water tomato paste water high fru...,halal
8,water tomato puree water tomato paste chicken ...,haram
9,prepared navy beans water brown sugar onion to...,halal


In [4]:
df.shape

(39787, 2)

In [5]:
df.columns

Index(['text', 'label'], dtype='object')

In [6]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [8]:
df['label'].value_counts()

label
halal    21826
haram    17961
Name: count, dtype: int64

In [10]:
df = df.drop_duplicates()

In [12]:
df["label"].value_counts()

label
halal    21612
haram    17961
Name: count, dtype: int64

In [13]:
df["label"].value_counts(normalize=True) * 100

label
halal    54.612994
haram    45.387006
Name: proportion, dtype: float64

In [14]:
# normalize label column 
df['label'] = df['label'].astype(str).str.strip().str.lower()

In [15]:
#keeping only halal haram labels
df = df[df['label'].isin(['halal', 'haram'])]
print("After filtering labels:", df.shape)
print(df['label'].value_counts())

After filtering labels: (39573, 2)
label
halal    21612
haram    17961
Name: count, dtype: int64


In [17]:
#cleaning text column 
import re
def clean_text(text):
    text = str(text).lower()                      
    text = re.sub(r'[^a-z\s]', ' ', text)          
    text = re.sub(r'\s+', ' ', text).strip()     
    return text
 
df['text'] = df['text'].apply(clean_text)

In [20]:
#label encoding
df['label_encoded'] = df['label'].map({'halal': 0, 'haram': 1})

In [21]:
#reset index and save the cleaned dataset
df = df.reset_index(drop=True)
df.to_csv("food_ingredients_cleaned.csv", index=False)

In [23]:
X = df['text']
y = df['label_encoded'] 

# Dataset Splitting

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [25]:
X_train.shape

(31658,)

In [26]:
X_test.shape

(7915,)

# Model A(Logistic Regression)

In [39]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Create the multi-hot encoder
vectorizer = CountVectorizer(binary=True)
X_train_encoded = vectorizer.fit_transform(X_train)
X_test_encoded = vectorizer.transform(X_test)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

Training shape: (31658, 9488)
Testing shape: (7915, 9488)


In [40]:
model_A = LogisticRegression(max_iter=1000)
model_A.fit(X_train_encoded, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [41]:
y_pred_A = model_A.predict(X_test_encoded)

In [63]:
#Full evaluation metrices
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n--- {name} ---")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))
    print("ROC-AUC  :", roc_auc_score(y_true, y_prob))
    print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["Halal", "Haram"]))

y_prob_A = model_A.predict_proba(X_test_encoded)[:, 1]
evaluate_model("Logistic Regression", y_test, y_pred_A, y_prob_A)


--- Logistic Regression ---
Accuracy : 0.9859759949463045
Precision: 0.9929198527329368
Recall   : 0.9760579064587973
F1 Score : 0.9844166783658571
ROC-AUC  : 0.9984485017467557

Confusion Matrix:
 [[4298   25]
 [  86 3506]]

Classification Report:
               precision    recall  f1-score   support

       Halal       0.98      0.99      0.99      4323
       Haram       0.99      0.98      0.98      3592

    accuracy                           0.99      7915
   macro avg       0.99      0.99      0.99      7915
weighted avg       0.99      0.99      0.99      7915



# Model B(LSTM)

In [45]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [46]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [47]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


In [65]:
lengths = [len(seq) for seq in X_train_seq]
print("Max length:", max(lengths))
print("Average length:", sum(lengths)/len(lengths))

Max length: 378
Average length: 42.2128371975488


In [66]:
maxlen = 50

In [67]:
X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

In [68]:
model_B = keras.Sequential([
    keras.layers.Input(shape=(maxlen,)),
    keras.layers.Embedding(input_dim=10000, output_dim=64),
    keras.layers.LSTM(64),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])
model_B.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_B.summary()


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 50, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 675,137 (2.58 MB)

 Trainable params: 675,137 (2.58 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
history_lstm = model_B.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256
)

Epoch 1/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 21s 152ms/step - accuracy: 0.8081 - loss: 0.4104 - val_accuracy: 0.9316 - val_loss: 0.1919
Epoch 2/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 143ms/step - accuracy: 0.9469 - loss: 0.1385 - val_accuracy: 0.9484 - val_loss: 0.1461
Epoch 3/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 15s 148ms/step - accuracy: 0.9605 - loss: 0.1053 - val_accuracy: 0.9547 - val_loss: 0.1155
Epoch 4/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 15s 147ms/step - accuracy: 0.9709 - loss: 0.0767 - val_accuracy: 0.9608 - val_loss: 0.1045
Epoch 5/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 20s 141ms/step - accuracy: 0.9750 - loss: 0.0660 - val_accuracy: 0.9613 - val_loss: 0.0940
Epoch 6/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 145ms/step - accuracy: 0.9795 - loss: 0.0573 - val_accuracy: 0.9585 - val_loss: 0.1139
Epoch 7/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 15s 148ms/step - accuracy: 0.9808 - loss: 0.0514 - val_accuracy: 0.9605 - val_loss: 0.1043
Epoch 8/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 144ms/step - accuracy: 0.9833 - loss: 0.0468 - val_accu

In [70]:
y_prob_B = model_B.predict(X_test_pad).ravel()   
y_pred_B = (y_prob_B >= 0.5).astype(int)        

evaluate_model("LSTM", y_test, y_pred_B, y_prob_B)

248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step

--- LSTM ---
Accuracy : 0.9641187618445989
Precision: 0.9633053221288516
Recall   : 0.9574053452115813
F1 Score : 0.9603462719910639
ROC-AUC  : 0.9930982734913013

Confusion Matrix:
 [[4192  131]
 [ 153 3439]]

Classification Report:
               precision    recall  f1-score   support

       Halal       0.96      0.97      0.97      4323
       Haram       0.96      0.96      0.96      3592

    accuracy                           0.96      7915
   macro avg       0.96      0.96      0.96      7915
weighted avg       0.96      0.96      0.96      7915



# Error Analysis

In [75]:
test_data = [
    ("chicken breast", 0),
    ("olive oil", 0),
    ("pork bacon", 1),
    ("alcohol", 1),
    ("gelatin", 1),
    ("vanilla extract", 1),
    ("lard substitute", 1),
    ("wine vinegar", 1),
    ("rennet", 1),
    ("porkk", 1),
    ("gelaitn", 1),
    ("sugar, gelatin, salt", 1),
    ("beef stock", 0),
    ("halal chicken broth", 0),
]

results = []

for ingredient, true_label in test_data:
    
    # Model A prediction
    vec = vectorizer.transform([ingredient])
    pred_a = model_A.predict(vec)[0]
    
    # Model B prediction
    seq = tokenizer.texts_to_sequences([ingredient])
    pad = pad_sequences(seq, maxlen=50, padding='post', truncating='post')
    prob_b = model_B.predict(pad)[0][0]
    pred_b = 1 if prob_b > 0.5 else 0
    correct_a = "Yes" if pred_a == true_label else "No"
    correct_b = "Yes" if pred_b == true_label else "No"
    
    results.append([ingredient, true_label, pred_a, correct_a, pred_b, correct_b])
    
    print(ingredient, "| True:", true_label, "| A:", pred_a, "| B:", pred_b)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
chicken breast | True: 0 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
olive oil | True: 0 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
pork bacon | True: 1 | A: 1 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
alcohol | True: 1 | A: 1 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
gelatin | True: 1 | A: 1 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
vanilla extract | True: 1 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
lard substitute | True: 1 | A: 0 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
wine vinegar | True: 1 | A: 1 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
rennet | True: 1 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
porkk | True: 1 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
gelaitn | True: 1 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
sugar, gelatin, salt | True: 1 | A: 1 | B: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
beef stock | True: 0 | A: 0 | B: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/s

In [76]:
df_results = pd.DataFrame(results, columns=["Ingredient", "True Label", "Model A Pred", "A Correct", "Model B Pred", "B Correct"])
df_results.to_csv("test_results.csv", index=False)

print(df_results)

              Ingredient  True Label  Model A Pred A Correct  Model B Pred  \
0         chicken breast           0             0       Yes             0   
1              olive oil           0             0       Yes             0   
2             pork bacon           1             1       Yes             1   
3                alcohol           1             1       Yes             1   
4                gelatin           1             1       Yes             1   
5        vanilla extract           1             0        No             0   
6        lard substitute           1             0        No             1   
7           wine vinegar           1             1       Yes             1   
8                 rennet           1             0        No             0   
9                  porkk           1             0        No             0   
10               gelaitn           1             0        No             0   
11  sugar, gelatin, salt           1             1       Yes    